# Applied Machine Learning, Notebook 10
## Detecting fraud with an autoencoder (Keras), and looking inside with t-SNE and UMAP

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we come from

In notebook 8 we built a fraud detector without labels: PCA learns the directions in which normal transactions
move, reconstructs each transaction with those directions, and the **reconstruction error** is the suspicion
score. We learned three things that today we take as given: the detector must be trained on **normal**
transactions (if fraud enters the training data, PCA learns it as one more pattern and stops seeing it), the
honest test is to train on the **past** and measure on the **future**, and in that test PCA reaches a PR-AUC of
0.59 with 76 frauds among the 100 most suspicious alerts.

PCA has a built-in limitation: it is **linear**. It can only reconstruct a transaction as a weighted sum of
fixed directions. If normal transactions live on a curved surface, PCA cannot follow it. Today's question:
**does a neural network that reconstructs, an autoencoder, detect better?**

### What you will learn today

1. What an **autoencoder** is: an encoder, a bottleneck and a decoder, trained to reproduce their own input.
2. Why a **linear** autoencoder is, deep down, a PCA, and how to check it.
3. How to build, train and use an autoencoder in **Keras**, layer by layer.
4. How to compare honestly: PCA, linear autoencoder, nonlinear autoencoder and the supervised models.
5. How to look at the compressed space (the **code** or *embedding*) with **t-SNE** and **UMAP**, and what
   cannot be concluded from those pictures.

In [1]:
# In Colab, TensorFlow and Keras come preinstalled; UMAP does not.
try:
    import umap
except ImportError:
    !pip install -q umap-learn
    import umap

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os, urllib.request, warnings, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
from tensorflow import keras

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
print("TensorFlow", tf.__version__)

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I0000 00:00:1790874746.529632   30699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790874746.575457   30699 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790874748.297676   30699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.21.0


## 1. The data and the preparation from notebook 8

The same credit card transaction dataset (284,807 transactions, of which 492 are fraud), the same 29 columns
(`V1` to `V28` and `Amount`) and the honest split: the first 70% of the time is training and the last 30% is
testing. We standardize with training, and the detector learns **only from the normal transactions** in
training.

In [2]:
URL_FRAUDE = ""    # <-- paste the link to the course CSV (creditcard.csv) here if you have it

def normalizar_link(url):
    '''Turns a Dropbox or Google Drive link into a direct download link.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if not os.path.exists("creditcard.csv"):
    fuente = normalizar_link(URL_FRAUDE) if URL_FRAUDE else \
        "https://huggingface.co/datasets/David-Egea/Creditcard-fraud-detection/resolve/main/creditcard.csv"
    print("Downloading creditcard.csv (150 MB, a couple of minutes)...")
    urllib.request.urlretrieve(fuente, "creditcard.csv")
datos = pd.read_csv("creditcard.csv")

COLS = [f"V{i}" for i in range(1, 29)] + ["Amount"]
orden = datos.sort_values("Time").reset_index(drop=True)
corte = int(len(orden) * 0.70)
tr, te = orden.iloc[:corte], orden.iloc[corte:]
y_tr, y_te = tr["Class"].values, te["Class"].values
escalador = StandardScaler().fit(tr[COLS])
X_tr, X_te = escalador.transform(tr[COLS]), escalador.transform(te[COLS])
X_normal = X_tr[y_tr == 0]                    # the detectors learn from this
print("training:", len(X_tr), "transactions (", len(X_normal), "normal ones to learn from )  |  testing:", len(X_te), "with", y_te.sum(), "frauds")

def evaluar(nombre, puntaje, y=y_te):
    orden_ = np.argsort(-puntaje)
    return {"modelo": nombre, "PR-AUC": average_precision_score(y, puntaje), "AUC ROC": roc_auc_score(y, puntaje),
            "precision@100": y[orden_][:100].mean(), "recall@500": y[orden_][:500].sum() / y.sum()}

pca = PCA(n_components=0.95).fit(X_normal)
K = int(pca.n_components_)
e_pca = ((X_te - pca.inverse_transform(pca.transform(X_te))) ** 2).sum(axis=1)
resultados = [evaluar(f"PCA (k = {K})", e_pca)]
pd.DataFrame(resultados).set_index("modelo").round(3)

training: 199364 transactions ( 198980 normal ones to learn from )  |  testing: 85443 with 108 frauds


,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
PCA (k = 26),0.589,0.888,0.76,0.778


That is the starting point: PCA with the 95% rule keeps $k = 26$ components and reaches a PR-AUC of 0.59.
Every new model today is added to the same `resultados` table.

## 2. What an autoencoder is

An **autoencoder** (Rumelhart, Hinton and Williams, 1986; Hinton and Salakhutdinov, 2006) is a neural network
trained for a seemingly absurd task: return exactly what it receives. The trick is in the middle: the
information has to pass through a narrow layer, the **bottleneck**, which forces it to be compressed.

$$ h = f(x), \qquad \hat{x} = g(h), \qquad \text{loss} = \frac{1}{n}\sum_{i=1}^{n} \lVert x_i - \hat{x}_i \rVert^2 $$

- $x$: a transaction, the vector of 29 standardized columns.
- $f$: the **encoder**, one or more layers that take $x$ to a shorter vector.
- $h$: the **code** (*embedding*, latent representation), with $k$ numbers, $k < 29$. It is the bottleneck.
- $g$: the **decoder**, which tries to reconstruct the 29 columns from the code.
- $\hat{x}$: the reconstruction. The loss is the **mean squared error** (MSE)
  between $x$ and $\hat{x}$, averaged over the $n$ training transactions. It is the same reconstruction error
  as in PCA.

Each dense layer (`Dense` in Keras) performs the usual operation, a linear combination followed by an
**activation function**:

$$ \text{layer}(u) = \sigma(A\, u + b) $$

- $\text{layer}(u)$: the output of the layer; $u$: the vector entering the layer; $A$: the
  weight matrix; $b$: the bias vector.
- $\sigma$: the activation. If it is the identity, the layer is linear. If it is the **ReLU**,
  $\sigma(t) = \max(0, t)$, the layer can bend the space, and the network can follow curved surfaces.

**The connection with PCA** (Baldi and Hornik, 1989). If all activations are the identity and the loss is the
MSE, the best autoencoder with a code of $k$ numbers reconstructs exactly like PCA with $k$ components: the code
spans the same subspace as the first $k$ principal directions (though not necessarily with the same axes). The
linear autoencoder adds nothing; the novelty lies in the nonlinear activations. We check this first, which also
lets us learn Keras without distractions.

## 3. Keras step by step: a linear autoencoder

Building a network in Keras takes four steps: **define** the layers, **compile** (choose the loss and the
optimizer), **train** (`fit`) and **use** (`predict`). Our linear autoencoder has an input layer of 29, a code
of $k = 26$ and an output of 29, with no activations.

In [3]:
keras.utils.set_random_seed(0)                                        # so the result can be reproduced
entrada = keras.Input(shape=(29,), name="transaccion")
codigo = keras.layers.Dense(K, activation=None, name="codigo")(entrada)            # encoder f
salida = keras.layers.Dense(29, activation=None, name="reconstruccion")(codigo)    # decoder g
ae_lineal = keras.Model(entrada, salida, name="autoencoder_lineal")
ae_lineal.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
ae_lineal.summary()

Model: "autoencoder_lineal"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ transaccion (InputLayer)        │ (None, 29)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ codigo (Dense)                  │ (None, 26)             │           780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruccion (Dense)          │ (None, 29)             │           783 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,563 (6.11 KB)

 Trainable params: 1,563 (6.11 KB)

 Non-trainable params: 0 (0.00 B)

The summary counts the parameters: the code layer has $29 \times 26$ weights plus 26 biases (780), and the
output layer $26 \times 29 + 29$ (783). To train, we pass `X_normal` **twice**, as input and as target: we want
the output to resemble the input. We hold out 10% for validation and use **early stopping**
(`EarlyStopping`): if the validation loss does not improve for 3 epochs, training stops and keeps the best
weights. An **epoch** is a full pass over the data; in each epoch the weights are adjusted with gradient
descent on batches of 256 transactions (the *mini-batch* from the regression notebooks).

In [4]:
parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
t0 = time.time()
hist = ae_lineal.fit(X_normal, X_normal, epochs=30, batch_size=256, validation_split=0.1, callbacks=[parar], verbose=0)
print(f"Trained in {time.time() - t0:.0f} seconds, {len(hist.history['loss'])} epochs")

fig = go.Figure()
fig.add_scatter(y=hist.history["loss"], name="training", mode="lines+markers")
fig.add_scatter(y=hist.history["val_loss"], name="validation", mode="lines+markers")
fig.update_layout(title="Linear autoencoder: loss (MSE) per epoch", xaxis_title="epoch", yaxis_title="MSE", height=350)
fig.show()

def error_ae(modelo, X):
    return ((X - modelo.predict(X, batch_size=4096, verbose=0)) ** 2).sum(axis=1)

e_lineal = error_ae(ae_lineal, X_te)
X_val = X_normal[-int(0.1 * len(X_normal)):]              # Keras validation_split=0.1 uses the last 10% of the rows
mse_pca = ((X_val - pca.inverse_transform(pca.transform(X_val))) ** 2).mean()        # mean per cell, like the Keras loss
print("Validation MSE (the last 10% of the normal ones):  linear autoencoder", round(min(hist.history["val_loss"]), 4), " | PCA", round(mse_pca, 4))
print("Correlation between the PCA score and the linear autoencoder score (in logs):", round(np.corrcoef(np.log(e_pca), np.log(e_lineal))[0, 1], 3))
resultados.append(evaluar(f"linear autoencoder (code {K})", e_lineal))
pd.DataFrame(resultados).set_index("modelo").round(3)

Trained in 16 seconds, 10 epochs


Validation MSE (the last 10% of the normal ones):  linear autoencoder 0.0637  | PCA 0.0602
Correlation between the PCA score and the linear autoencoder score (in logs): 0.891


,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
PCA (k = 26),0.589,0.888,0.76,0.778
linear autoencoder (code 26),0.579,0.868,0.76,0.769


The validation loss of the linear autoencoder (0.064 per cell) lands practically on that of PCA on those same
transactions (0.060), and the detector is almost the same: PR-AUC of 0.58 versus 0.59, the same precision@100,
and highly correlated scores between the two versions. What Baldi and Hornik show in theory is visible in the
data. The difference from PCA is one of procedure, not of result: PCA finds the exact solution with algebra in
an instant, while the network searches for it with gradients over several epochs and gets close.

## 4. A nonlinear autoencoder

Now the version that can bend the space: a hidden layer of 64 neurons with ReLU before and after the code,
which we keep at $k = 26$ numbers to compare with PCA on equal terms.

| layer | input | hidden | code | hidden | output |
|---|---|---|---|---|---|
| neurons | 29 | 64 (ReLU) | 26 | 64 (ReLU) | 29 |

Nonlinear networks have a problem that PCA does not: the result depends on the weights they start from, which
are random. That is why we train **three times**, with three seeds, and report all three.
(In Colab without a GPU each training run takes about a minute; your numbers may differ from those in the text
in the second decimal, due to version differences.)

In [5]:
def autoencoder(k, ocultas=(64,), activacion="relu", semilla=0, l2=0.0):
    keras.utils.set_random_seed(semilla)
    reg = keras.regularizers.l2(l2) if l2 else None
    entrada = keras.Input(shape=(29,))
    capa = entrada
    for n in ocultas:                                                 # encoder
        capa = keras.layers.Dense(n, activation=activacion, kernel_regularizer=reg)(capa)
    codigo = keras.layers.Dense(k, name="codigo")(capa)
    capa = codigo
    for n in reversed(ocultas):                                       # decoder (mirror image)
        capa = keras.layers.Dense(n, activation=activacion, kernel_regularizer=reg)(capa)
    salida = keras.layers.Dense(29)(capa)
    modelo = keras.Model(entrada, salida)
    modelo.compile(optimizer="adam", loss="mse")
    return modelo

def entrenar(modelo, X):
    parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    return modelo.fit(X, X, epochs=30, batch_size=256, validation_split=0.1, callbacks=[parar], verbose=0)

no_lineales = {}
for semilla in [0, 1, 2]:
    t0 = time.time()
    m = autoencoder(K, ocultas=(64,), semilla=semilla)
    h = entrenar(m, X_normal)
    no_lineales[semilla] = m
    e = error_ae(m, X_te)
    resultados.append(evaluar(f"nonlinear autoencoder 64-{K}-64, seed {semilla}", e))
    print(f"seed {semilla}: {len(h.history['loss'])} epochs, {time.time() - t0:.0f} s, validation MSE {min(h.history['val_loss']):.4f}")
pd.DataFrame(resultados).set_index("modelo").round(3)

seed 0: 25 epochs, 44 s, validation MSE 0.0008


seed 1: 30 epochs, 49 s, validation MSE 0.0016


seed 2: 19 epochs, 33 s, validation MSE 0.0019


,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
PCA (k = 26),0.589,0.888,0.76,0.778
linear autoencoder (code 26),0.579,0.868,0.76,0.769
"nonlinear autoencoder 64-26-64, seed 0",0.624,0.944,0.78,0.769
"nonlinear autoencoder 64-26-64, seed 1",0.411,0.945,0.57,0.778
"nonlinear autoencoder 64-26-64, seed 2",0.541,0.947,0.68,0.778


Three readings, and the third is the important one.

1. **It reconstructs much better.** Its validation MSE lies between 0.001 and 0.002 per cell, dozens of times
   lower than PCA or the linear autoencoder (0.06): the activations let it follow normal transactions closely.
2. **It ranks the whole set better**: its AUC ROC is 0.94 for all three seeds, versus 0.89 for PCA.
3. **But it does not detect better where it matters.** The PR-AUC ranges from 0.41 to 0.62 depending on the
   seed (average 0.53, below PCA's 0.59), and the precision@100 from 0.57 to 0.78. One seed beats PCA; another
   falls clearly behind.

Why does better reconstruction not help? Because the detector does not need to reconstruct **everything** well:
it needs to reconstruct the normal well and **the fraud badly**. A more flexible network also learns to
reconstruct better combinations it never saw, including part of the fraud, and the contrast shrinks. With PCA,
the fraud direction (the one we found in notebook 8, `V17`, `V12`, `V14`, `V10`) falls cleanly outside the
subspace; in the network, it depends on where training took it.

It is the lesson of notebooks 2 and 4 in another disguise: **more complexity is not more information**. In this
dataset the pattern that separates fraud is almost linear, and the linear model captures it more stably.

## 5. Looking inside: the code, with t-SNE and UMAP

The encoder compresses each transaction into 26 numbers. Do the frauds end up in a corner of that space? To
see it we have to go down from 26 to 2 dimensions, and there are three ways:

- **PCA to 2 dimensions**: linear; it preserves large distances and variance, but it can pile everything up.
- **t-SNE** (van der Maaten and Hinton, 2008): looks for a 2D picture where close neighbors remain neighbors.
  Its main parameter, the *perplexity*, is roughly how many neighbors each point looks at.
- **UMAP** (McInnes, Healy and Melville, 2018): the same idea with different mathematics; faster, it preserves
  the global structure somewhat better, and unlike t-SNE it can apply the map to new data (`transform`).

We extract the code with an **encoder model**: a `keras.Model` that goes from the input to the `codigo` layer
of the already trained autoencoder. We use a sample of 3,000 normal testing transactions plus all the testing
frauds, because t-SNE and UMAP are slow with hundreds of thousands of points.

In [6]:
mejor = no_lineales[0]
codificador = keras.Model(mejor.input, mejor.get_layer("codigo").output)

rng = np.random.default_rng(0)
idx = np.r_[rng.choice(np.where(y_te == 0)[0], 3000, replace=False), np.where(y_te == 1)[0]]
H = codificador.predict(X_te[idx], verbose=0)               # 3,108 x 26: the code of each transaction
etiqueta = np.where(y_te[idx] == 1, "fraud", "normal")
print("Shape of the code:", H.shape)

t0 = time.time(); mapa_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(H); t_tsne = time.time() - t0
t0 = time.time(); mapa_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(H); t_umap = time.time() - t0
mapa_pca = PCA(n_components=2).fit_transform(H)
print(f"t-SNE: {t_tsne:.0f} s   UMAP: {t_umap:.0f} s")

mapas = pd.concat([pd.DataFrame({"x": m[:, 0], "y": m[:, 1], "class": etiqueta, "method": n})
                   for n, m in [("PCA to 2D", mapa_pca), ("t-SNE", mapa_tsne), ("UMAP", mapa_umap)]])
fig = px.scatter(mapas, x="x", y="y", color="class", facet_col="method", opacity=0.6,
                 color_discrete_map={"normal": "#94a3b8", "fraud": "#dc2626"},
                 title="The autoencoder code (26 dimensions) drawn in 2D: 3,000 normal and 108 fraudulent testing transactions")
fig.update_xaxes(matches=None, showticklabels=False, title=""); fig.update_yaxes(matches=None, showticklabels=False, title="")
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=430)
fig.show()

Shape of the code: (3108, 26)


t-SNE: 10 s   UMAP: 19 s


In PCA to 2D the frauds look like a tail coming out of the cloud of normal transactions: magnitude separates
them, but most normal transactions are piled up at one point. t-SNE and UMAP open up the cloud and show the
local structure better: a good share of the frauds forms one or two **groups of their own**, and another share
stays mixed among the normal ones. That mixed share is the fraud that no reconstruction detector will find: in
the space the network learned, those transactions look normal.

### What cannot be concluded from a t-SNE

These pictures are useful for exploring and dangerous for concluding. Three rules:

1. **Distances between groups mean nothing**, nor do group sizes: t-SNE enlarges dense regions and shrinks
   sparse ones.
2. **The parameters change the picture**: a different *perplexity* or a different seed gives a different map.
3. **t-SNE can invent groups where there are none.** Let us check: 3,000 points of pure Gaussian noise, with no
   structure at all, in 26 dimensions.

In [7]:
ruido = np.random.default_rng(1).normal(size=(3000, 26))
dibujos = pd.concat([pd.DataFrame({"x": m[:, 0], "y": m[:, 1], "method": n}) for n, m in [
    ("PCA to 2D", PCA(n_components=2).fit_transform(ruido)),
    ("t-SNE, perplexity 5", TSNE(n_components=2, perplexity=5, init="pca", random_state=0).fit_transform(ruido)),
    ("t-SNE, perplexity 50", TSNE(n_components=2, perplexity=50, init="pca", random_state=0).fit_transform(ruido))]])
fig = px.scatter(dibujos, x="x", y="y", facet_col="method", opacity=0.5, title="Pure noise, with no groups, seen by PCA and by t-SNE")
fig.update_xaxes(matches=None, showticklabels=False, title=""); fig.update_yaxes(matches=None, showticklabels=False, title="")
fig.update_traces(marker=dict(size=3, color="#64748b"))
fig.update_layout(height=380)
fig.show()

PCA shows what is there: a round cloud, with no groups. t-SNE with a low *perplexity* draws clumps and filaments
that anyone would interpret as segments, and with a high *perplexity* a different texture: both are the same
data with no structure. A group that appears in t-SNE or UMAP is a **hypothesis**; it is confirmed with
something else (the labels, as here, or an analysis that does not depend on the picture).

## 6. The final table: everyone against everyone

We add the two supervised models from notebook 8, trained with the training labels (all of them), on the same
time-based split.

In [8]:
s_log = LogisticRegression(max_iter=2000).fit(X_tr, y_tr).decision_function(X_te)
s_gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0,
                                      min_samples_leaf=50, random_state=0).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
resultados.append(evaluar("logistic (supervised)", s_log))
resultados.append(evaluar("gradient boosting (supervised)", s_gb))
final = pd.DataFrame(resultados).set_index("modelo")
nl = final.loc[final.index.str.startswith("nonlinear autoencoder")]
print("Nonlinear autoencoder, average of the three seeds:", nl.mean().round(3).to_dict())
final.round(3)

Nonlinear autoencoder, average of the three seeds: {'PR-AUC': 0.525, 'AUC ROC': 0.945, 'precision@100': 0.677, 'recall@500': 0.775}


,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
PCA (k = 26),0.589,0.888,0.76,0.778
linear autoencoder (code 26),0.579,0.868,0.76,0.769
"nonlinear autoencoder 64-26-64, seed 0",0.624,0.944,0.78,0.769
"nonlinear autoencoder 64-26-64, seed 1",0.411,0.945,0.57,0.778
"nonlinear autoencoder 64-26-64, seed 2",0.541,0.947,0.68,0.778
logistic (supervised),0.705,0.973,0.72,0.843
gradient boosting (supervised),0.759,0.979,0.81,0.815


In [9]:
fig = go.Figure()
for nombre, s in [(f"PCA (k = {K})", e_pca), ("linear autoencoder", e_lineal),
                  ("nonlinear autoencoder, seed 0", error_ae(no_lineales[0], X_te)),
                  ("logistic (supervised)", s_log), ("gradient boosting (supervised)", s_gb)]:
    p, r, _ = precision_recall_curve(y_te, s)
    fig.add_scatter(x=r, y=p, mode="lines", name=f"{nombre} ({average_precision_score(y_te, s):.2f})")
fig.update_layout(title="Precision-recall curves on the future; in parentheses, the PR-AUC", xaxis_title="recall",
                  yaxis_title="precision", height=450)
fig.show()

## 7. Key takeaways

1. An **autoencoder** is a network trained to reproduce its input through a bottleneck; the reconstruction
   error works as an anomaly score exactly as in PCA.
2. A **linear** autoencoder with MSE loss is a PCA: same reconstruction, same detector, slower. The novelty
   lies in the nonlinear activations.
3. In Keras: `Input` and `Dense` to define, `Model` to assemble, `compile` (optimizer and loss), `fit(X, X)`
   with validation and early stopping, `predict` to reconstruct, and a second `Model` up to the code layer to
   extract the *embedding*.
4. In this dataset the nonlinear autoencoder **reconstructs better but does not detect better**, and its result
   depends on the seed. Reconstructing the normal well and the anomalous badly is not the same as
   reconstructing everything well.
5. t-SNE and UMAP are useful to **look at** a high-dimensional space; they are not useful to **conclude**: they
   invent groups in pure noise and their distances mean nothing.
6. With all the labels, supervised learning still wins. The value of unsupervised detectors lies in fraud that
   does not have labels yet, and in the first days of a new system.

## Exercises

1. **The size of the bottleneck.** Train the nonlinear autoencoder (64 hidden neurons, seed 0) with codes of 8
   and of 15 numbers, and compare their PR-AUC with that of the code of 26. Before running, predict the result
   using what you learned in notebook 8 about which components carry the fraud signal.
2. **Contamination, again.** Train the nonlinear autoencoder (code 26, seed 0) with **all** the training
   transactions, frauds included, instead of only the normal ones. Does the same thing happen to it as to PCA
   in notebook 8?
3. **Combining the two detectors.** PCA and the autoencoder make mistakes on different transactions. Build a
   combined score by averaging the **ranks** of each transaction according to PCA and according to the
   nonlinear autoencoder with seed 0 (rank 1 is the least suspicious). Does the PR-AUC or the precision@100
   improve relative to each one separately? Why do we average ranks and not the errors directly?

## Solutions

In [10]:
# Exercise 1: the size of the bottleneck
filas = []
for k in [8, 15]:
    m = autoencoder(k, ocultas=(64,), semilla=0)
    entrenar(m, X_normal)
    filas.append(evaluar(f"nonlinear, code {k}", error_ae(m, X_te)))
filas.append(evaluar(f"nonlinear, code {K}", error_ae(no_lineales[0], X_te)))
pd.DataFrame(filas).set_index("modelo").round(3)

,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
"nonlinear, code 8",0.216,0.918,0.36,0.694
"nonlinear, code 15",0.345,0.925,0.45,0.778
"nonlinear, code 26",0.624,0.944,0.78,0.769


With a narrow code the detector gets much worse. In notebook 8 we saw that the fraud signal lives in a
direction where normal transactions barely vary: one of the **last** components. A code of 8 or 15 numbers
forces the network to discard many directions that do vary among normal transactions, and the reconstruction
error of any transaction becomes dominated by that unreconstructed normal variation, not by rarity. It is the
same effect as PCA with a small $k$ (section 7.1 of notebook 8). In anomaly detection the bottleneck has to be
**wide**: discard little, and what is discarded has to be what normal transactions do not use. It is the
opposite of what you look for when you want to compress for visualization.

In [11]:
# Exercise 2: training with all of training (frauds included)
m_todo = autoencoder(K, ocultas=(64,), semilla=0)
entrenar(m_todo, X_tr)
pd.DataFrame([evaluar("nonlinear, trained only on normal transactions", error_ae(no_lineales[0], X_te)),
              evaluar("nonlinear, trained on all of training", error_ae(m_todo, X_te))]).set_index("modelo").round(3)

,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
"nonlinear, trained only on normal transactions",0.624,0.944,0.78,0.769
"nonlinear, trained on all of training",0.243,0.938,0.32,0.778


The autoencoder also suffers from contamination: the PR-AUC drops from 0.62 to 0.24 and the precision@100 from
0.78 to 0.32. The reason is the same as with PCA: the frauds of the first day are few but extreme, the network
reduces its loss by learning to reconstruct them, and stops seeing them as rare. The deterioration is somewhat
less abrupt than with PCA (which fell to zero), but the rule does not change: whatever the detector sees in
training is, for it, normal.

In [12]:
# Exercise 3: combining the two detectors through ranks
from scipy.stats import rankdata
e_nl0 = error_ae(no_lineales[0], X_te)
combinado = (rankdata(e_pca) + rankdata(e_nl0)) / 2
pd.DataFrame([evaluar(f"PCA (k = {K})", e_pca), evaluar("nonlinear autoencoder, seed 0", e_nl0),
              evaluar("rank average", combinado)]).set_index("modelo").round(3)

,PR-AUC,AUC ROC,precision@100,recall@500
modelo,,,,
PCA (k = 26),0.589,0.888,0.76,0.778
"nonlinear autoencoder, seed 0",0.624,0.944,0.78,0.769
rank average,0.720,0.922,0.83,0.787


We average **ranks** because the two errors are on very different scales (the autoencoder's is dozens of times
smaller, because it reconstructs better), and averaging the raw values would give all the weight to PCA for no
reason. Ranks put both on the same scale, from 1 to 85,443.

The average beats both separately: PR-AUC of 0.72 versus 0.59 and 0.62, and 83 frauds among the first 100
alerts. The two detectors make mistakes on different transactions: PCA is strong in the linear direction of
fraud, and the network ranks the rest better (its AUC ROC is higher). Together they cover each other. This holds
for this seed; with a worse seed the average gains less, but it still covers the network's bad training runs.
It is the idea of the ensembles from notebook 6 applied to unsupervised detectors.